# 📦 RMUC 消息拆分说明文档

> **项目**: RoboMaster RMUC 2026 哨兵行为树  
> **日期**: 2026-02-25（最后更新 2026-07）  
> **分支**: `test`

---

### 背景

原始设计中，所有 RMUC 比赛数据封装在一个 **272 行的 `RMUC.msg`** 大消息中，通过统一的 `/rmuc` 话题传输。这带来了三个问题：

| 问题 | 影响 |
|:---|:---|
| **频率耦合** | 1 Hz 的比赛状态和 50 Hz 的位姿数据被迫以相同频率发布 |
| **带宽浪费** | 每帧都传输完整消息，即使只有少数字段更新 |
| **调试困难** | `ros2 topic echo` 输出刷屏，难以定位单一数据源问题 |

**改进方案**：将 `RMUC.msg` 拆分为 **15 个独立的小消息**（12 📥 订阅 + 3 📤 发布），各自拥有独立话题、独立频率，实现数据解耦。

> ⚠️ **P0 更新 (2026-07)**：新增 7 个 RMUC 2026 裁判系统协议话题，订阅总数从 5 → **12**。

```
原架构:  [所有数据] ──→ /rmuc (单话题)
新架构:  [比赛状态]     ──→ /game_status              (1 Hz)
         [机器人状态]   ──→ /robot_status             (10 Hz)
         [RFID]         ──→ /rfid_status              (事件驱动)
         [位姿]         ──→ /robot_position           (50 Hz)
         [雷达]         ──→ /radar/enemy_tracks       (10-30 Hz)
         [哨兵决策状态] ──→ /sentry_decision_status   (10 Hz)  ← NEW
         [机器人增益]   ──→ /robot_buff               (10 Hz)  ← NEW
         [弹丸配额]     ──→ /projectile_allowance     (10 Hz)  ← NEW
         [场地状态]     ──→ /field_status             (1 Hz)   ← NEW
         [敌方易伤]     ──→ /enemy_mark               (1 Hz)   ← NEW
         [队友位置]     ──→ /team_positions           (1 Hz)   ← NEW
         [队伍血量]     ──→ /team_hp                  (10 Hz)  ← NEW
         [决策指令]     ←── /sentry_cmd               (2 Hz)
         [云台控制]     ←── /robot_control            (10 Hz)
         [导航控制]     ←── /nav_control_cmd          (按需)
```

## 1. 拆分后的 15 个消息详情

> 所有 `.msg` 文件位于 `rm_decision_interfaces/msg/RMUC/` 目录下。  
> 箭头方向：`→` 表示 BT 订阅（输入），`←` 表示 BT 发布（输出）。  
> 🆕 标记为 RMUC 2026 新增话题。

---

### 1.1 📥 RMUCGameStatus — 比赛状态
| 属性 | 值 |
|:---|:---|
| **话题** | `/game_status` |
| **BT 节点** | `RmucSubGameStatus` |
| **频率** | 1 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `game_status` (raw msg), `now_ms` (uint64, 毫秒时间戳) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `game_progress` | `uint8` | 比赛阶段 (0=未开始, 4=进行中, 5=结算) |
| `stage_remain_time` | `uint16` | 当前阶段剩余时间 (秒) |
| red/blue team data | — | 红/蓝方队伍数据 |

---

### 1.2 📥 RMUCRobotStatus — 机器人核心状态
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_status` |
| **BT 节点** | `RmucSubRobotStatus` |
| **频率** | 10 Hz |
| **方向** | 电控 → BT |
| **输出端口** | `robot_status` (raw msg) |

| 字段分组 | 包含字段 |
|:---|:---|
| 血量与热量 | `hp`, `max_hp`, `heat`, `max_heat`, `cooling_rate` |
| 弹丸 | `ammo` |
| 功率 | `shooter_power_output`, `chassis_power` |
| 状态标志 | `is_dead`, `is_disengaged`, `disengage_cd_s` |

---

### 1.3 📥 RMUCRFIDStatus — RFID 场地交互
| 属性 | 值 |
|:---|:---|
| **话题** | `/rfid_status` |
| **BT 节点** | `RmucSubRFIDStatus` |
| **频率** | 事件驱动 (变化时发布) |
| **方向** | 电控 → BT |
| **输出端口** | `rfid_status` (raw msg) |

内容：`base_buff`, `outpost_buff`, 各增益点 RFID 检测状态

---

### 1.4 📥 RMUCRobotPosition — 自身位姿
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_position` |
| **BT 节点** | `RmucSubRobotPosition` |
| **频率** | 50 Hz |
| **方向** | 定位系统 → BT |
| **输出端口** | `pose_x`, `pose_y`, `pose_yaw`, `is_at_nav_goal`, `pose` |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `pose_x` | `float32` | 地图坐标系 x (m) |
| `pose_y` | `float32` | 地图坐标系 y (m) |
| `pose_yaw` | `float32` | 航向角 (rad) |
| `is_at_nav_goal` | `bool` | 是否已到达导航目标点 |
| `pose` | `geometry_msgs/Pose` | 完整位姿 |

---

### 1.5 📥 RMUCEnemyTracks — 雷达敌方跟踪
| 属性 | 值 |
|:---|:---|
| **话题** | `/radar/enemy_tracks` |
| **BT 节点** | `SubRadarTracks` |
| **频率** | 10-30 Hz |
| **方向** | 雷达站 → BT |
| **输出端口** | `radar_tracks` (raw msg) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `enemy_count` | `uint8` | 有效跟踪数 (0~7) |
| `enemy_robot_id[]` | `uint8[]` | 敌方机器人 ID |
| `enemy_x[]` / `enemy_y[]` | `float32[]` | 地图坐标位置 |
| `enemy_confidence[]` | `float32[]` | 置信度 [0, 1] |

---

### 1.6 📥🆕 RMUCSentryDecisionStatus — 哨兵决策状态 (`0x020D`)
| 属性 | 值 |
|:---|:---|
| **话题** | `/sentry_decision_status` |
| **BT 节点** | `RmucSubSentryDecisionStatus` |
| **频率** | 10 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `sentry_decision_status` (raw msg) |

| 字段分组 | 包含字段 | 说明 |
|:---|:---|:---|
| 复活状态 | `can_free_respawn`, `can_instant_respawn`, `instant_respawn_cost` | 免费/立即复活可用性与花费 |
| 姿态反馈 | `current_posture` | 当前执行姿态 |
| 遥控兑换 | `remote_ammo_count`, `remote_heal_count`, `exchanged_ammo_total` | 遥控补弹/补血次数及累计兑换弹量 |
| 能量机关 | `can_activate_energy` | 能量机关激活权 |

---

### 1.7 📥🆕 RMUCRobotBuff — 机器人增益状态 (`0x0204`)
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_buff` |
| **BT 节点** | `RmucSubRobotBuff` |
| **频率** | 10 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `robot_buff` (raw msg) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `buff_heal_rate` | `float32` | 回血速率 |
| `buff_cool_value` | `float32` | 冷却值 |
| `buff_defense_pct` | `float32` | 防御加成 (%) |
| `buff_vulnerability_pct` | `float32` | 易伤加成 (%) |
| `buff_attack_pct` | `float32` | 攻击加成 (%) |

---

### 1.8 📥🆕 RMUCProjectileAllowance — 弹丸配额 (`0x0208`)
| 属性 | 值 |
|:---|:---|
| **话题** | `/projectile_allowance` |
| **BT 节点** | `RmucSubProjectileAllowance` |
| **频率** | 10 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `projectile_allowance` (raw msg) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `fortress_ammo` | `uint16` | 前哨站存弹量 |

---

### 1.9 📥🆕 RMUCFieldStatus — 场地占领状态 (`0x0101`)
| 属性 | 值 |
|:---|:---|
| **话题** | `/field_status` |
| **BT 节点** | `RmucSubFieldStatus` |
| **频率** | 1 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `field_status` (raw msg) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `field_central_highland` | `uint8` | 中央高地占领/归属状态 |
| `field_ladder_highland` | `uint8` | 梯形高地占领/归属状态 |
| `field_fortress` | `uint8` | 堡垒占领/归属状态 |
| `field_outpost_buff` | `uint8` | 前哨增益点状态 |
| `field_base_buff` | `uint8` | 基地增益点状态 |
| `field_small_energy` | `uint8` | 小能量机关状态 |
| `field_big_energy` | `uint8` | 大能量机关状态 |

---

### 1.10 📥🆕 RMUCEnemyMark — 敌方易伤标记 (`0x020C`)
| 属性 | 值 |
|:---|:---|
| **话题** | `/enemy_mark` |
| **BT 节点** | `RmucSubEnemyMark` |
| **频率** | 1 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `enemy_mark` (raw msg) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `enemy_hero_vuln` | `uint8` | 敌方英雄易伤标记 |
| `enemy_engi_vuln` | `uint8` | 敌方工程易伤标记 |
| `enemy_infantry3_vuln` | `uint8` | 敌方步兵3号易伤标记 |
| `enemy_infantry4_vuln` | `uint8` | 敌方步兵4号易伤标记 |
| `enemy_sentry_vuln` | `uint8` | 敌方哨兵易伤标记 |

---

### 1.11 📥🆕 RMUCTeamPositions — 队友位置
| 属性 | 值 |
|:---|:---|
| **话题** | `/team_positions` |
| **BT 节点** | `RmucSubTeamPositions` |
| **频率** | 1 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `team_positions` (raw msg) |

内容：己方所有机器人位置信息（广播给队友）

---

### 1.12 📥🆕 RMUCTeamHP — 队伍血量 (`0x0003`)
| 属性 | 值 |
|:---|:---|
| **话题** | `/team_hp` |
| **BT 节点** | `RmucSubTeamHP` |
| **频率** | 10 Hz |
| **方向** | 裁判系统 → BT |
| **输出端口** | `team_hp` (raw msg) |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `team_outpost_hp` | `uint16` | 前哨站血量 |
| `team_base_hp` | `uint16` | 基地血量 |

---

### 1.13 📤 RMUCSentryCmd — 哨兵决策指令
| 属性 | 值 |
|:---|:---|
| **话题** | `/sentry_cmd` |
| **频率** | 2 Hz |
| **方向** | BT → 电控 |

| 字段 | 说明 |
|:---|:---|
| `cmd_posture` | 姿态指令 (1=进攻, 2=防御, 3=移动) |
| `cmd_confirm_respawn` | 确认复活 |
| `cmd_confirm_instant_respawn` | 确认立即复活 |
| `cmd_allow_ammo_target` | 允许发弹量目标 |
| `cmd_trigger_remote_ammo` | 远程补弹 (上升沿) |
| `cmd_trigger_remote_hp` | 远程补血 (上升沿) |
| `cmd_enable_big_energy` | 大能量机关确认 |

---

### 1.14 📤 RMUCRobotControl — 云台与底盘控制
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_control` |
| **频率** | 10 Hz |
| **方向** | BT → 电控 |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `stop_gimbal_scan` | `bool` | 停止云台扫描 |
| `chassis_spin` | `bool` | 底盘小陀螺 |
| `fire_enable` | `bool` | 允许发射 |

---

### 1.15 📤 RMUCNavControlCmd — 导航控制
| 属性 | 值 |
|:---|:---|
| **话题** | `/nav_control_cmd` |
| **频率** | 按需发布 |
| **方向** | BT → 电控 |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `cmd_type` | `int32` | 0=无操作, 1=开始, 2=终止, 3=原地 |
| `emergency_stop` | `bool` | 紧急停止 |

## 2. 话题映射表 (旧 → 新)

> 所有话题名使用**绝对路径**（如 `/game_status` 而非 `game_status`）。  
> 🆕 = RMUC 2026 新增话题。

| 方向 | 原 `/rmuc` 字段类别 | 新消息类型 | 独立话题 | BT 插件 | 频率 |
|:---:|:---|:---|:---|:---|:---:|
| 📥 | Game Status | `RMUCGameStatus` | `/game_status` | `RmucSubGameStatus` | 1 Hz |
| 📥 | Robot HP / Heat / Ammo | `RMUCRobotStatus` | `/robot_status` | `RmucSubRobotStatus` | 10 Hz |
| 📥 | RFID Flags | `RMUCRFIDStatus` | `/rfid_status` | `RmucSubRFIDStatus` | 事件 |
| 📥 | Robot Pose | `RMUCRobotPosition` | `/robot_position` | `RmucSubRobotPosition` | 50 Hz |
| 📥 | Radar Tracks | `RMUCEnemyTracks` | `/radar/enemy_tracks` | `SubRadarTracks` | 10-30 Hz |
| 📥🆕 | Sentry Decision (0x020D) | `RMUCSentryDecisionStatus` | `/sentry_decision_status` | `RmucSubSentryDecisionStatus` | 10 Hz |
| 📥🆕 | Robot Buff (0x0204) | `RMUCRobotBuff` | `/robot_buff` | `RmucSubRobotBuff` | 10 Hz |
| 📥🆕 | Projectile Allowance (0x0208) | `RMUCProjectileAllowance` | `/projectile_allowance` | `RmucSubProjectileAllowance` | 10 Hz |
| 📥🆕 | Field Status (0x0101) | `RMUCFieldStatus` | `/field_status` | `RmucSubFieldStatus` | 1 Hz |
| 📥🆕 | Enemy Mark (0x020C) | `RMUCEnemyMark` | `/enemy_mark` | `RmucSubEnemyMark` | 1 Hz |
| 📥🆕 | Team Positions | `RMUCTeamPositions` | `/team_positions` | `RmucSubTeamPositions` | 1 Hz |
| 📥🆕 | Team HP (0x0003) | `RMUCTeamHP` | `/team_hp` | `RmucSubTeamHP` | 10 Hz |
| 📤 | Sentry Decision | `RMUCSentryCmd` | `/sentry_cmd` | `SentryCmdMux` | 2 Hz |
| 📤 | Gimbal / Chassis | `RMUCRobotControl` | `/robot_control` | `RmucRobotControl` | 10 Hz |
| 📤 | Nav Control | `RMUCNavControlCmd` | `/nav_control_cmd` | `RmucNavControlCmd` | 按需 |

## 3. 插件引用关系

> 📌 **图例**：🔵 订阅者 · 🟢 条件节点 · 🟠 动作节点 · 🔴 解析/混合

---

### `RMUCGameStatus` → 3 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubGameStatus` | 🔵 | 订阅 `/game_status`，写入黑板 `{game_status}` + `{now_ms}` |
| `IsGameTime` | 🟢 | 判断当前比赛阶段是否为指定值 |
| `ParseSentryBlackboard` | 🔴 | 解析 `stage_remain`, `elapsed_time` 等派生变量 |

### `RMUCRobotStatus` → 9 个插件 ⭐ 引用最多
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRobotStatus` | 🔵 | 订阅 `/robot_status`，写入黑板 `{robot_status}` |
| `DetectRespawnAndSetRecovery` | 🔴 | 订阅 + 检测复活沿 |
| `WaitAndHeal` | 🔴 | 订阅 + 等待回血到指定比例 |
| `IsDead` | 🟢 | 判断 `is_dead` |
| `IsHPBelow` | 🟢 | 判断 `current_hp < threshold` |
| `DecideRespawnCmd` | 🟠 | 读取复活状态生成决策指令 |
| `ParseSentryBlackboard` | 🔴 | 解析 HP/弹丸/经济等派生变量 |
| `IsZoneCardDetected` | 🟢 | 混合引用 (同时读 RFID) |
| `IsAnyDispelCardDetected` | 🟢 | 混合引用 (同时读 RFID) |

### `RMUCRFIDStatus` → 5 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRFIDStatus` | 🔵 | 订阅 `/rfid_status`，写入黑板 `{rfid_status}` |
| `IsSupplyCardDetected` | 🟢 | 判断 `rfid_supply` |
| `IsZoneCardDetected` | 🟢 | 判断多个增益区 RFID |
| `IsAnyDispelCardDetected` | 🟢 | 判断是否触发任意驱散点 |
| `MicroSearchSupplyCard` | 🟠 | 微调搜索补给卡 |

### `RMUCRobotPosition` → 2 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRobotPosition` | 🔵 | 订阅 `/robot_position`，输出 `{pose_x/y/yaw}` + `{is_at_nav_goal}` + `{pose}` |
| `IsAtNavGoal` | 🟢 | 读取 `bool is_at_nav_goal` 黑板变量 |

### `RMUCEnemyTracks` → 3 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `SubRadarTracks` | 🔵 | 订阅 `/radar/enemy_tracks`，写入黑板 `{radar_tracks}` |
| `SelectBestTarget` | 🟠 | 根据距离/置信度选择最优打击目标 |
| `ParseSentryBlackboard` | 🔴 | 解析 `has_target` / `best_target` 等 |

---

### 🆕 以下为 RMUC 2026 新增话题的插件引用

### `RMUCSentryDecisionStatus` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubSentryDecisionStatus` | 🔵 | 订阅 `/sentry_decision_status`，写入黑板 `{sentry_decision_status}` |
| `ParseSentryBlackboard` | 🔴 | 解析复活/姿态/遥控兑换/能量机关等派生变量 |

### `RMUCRobotBuff` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRobotBuff` | 🔵 | 订阅 `/robot_buff`，写入黑板 `{robot_buff}` |
| `ParseSentryBlackboard` | 🔴 | 解析增益速率/冷却/防御/易伤/攻击等派生变量 |

### `RMUCProjectileAllowance` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubProjectileAllowance` | 🔵 | 订阅 `/projectile_allowance`，写入黑板 `{projectile_allowance}` |
| `ParseSentryBlackboard` | 🔴 | 解析 `fortress_ammo` 等派生变量 |

### `RMUCFieldStatus` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubFieldStatus` | 🔵 | 订阅 `/field_status`，写入黑板 `{field_status}` |
| `ParseSentryBlackboard` | 🔴 | 解析各场地占领/归属状态派生变量 |

### `RMUCEnemyMark` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubEnemyMark` | 🔵 | 订阅 `/enemy_mark`，写入黑板 `{enemy_mark}` |
| `ParseSentryBlackboard` | 🔴 | 解析各敌方机器人易伤标记 |

### `RMUCTeamPositions` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubTeamPositions` | 🔵 | 订阅 `/team_positions`，写入黑板 `{team_positions}` |
| `ParseSentryBlackboard` | 🔴 | 广播己方机器人位置信息 |

### `RMUCTeamHP` → 1+ 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubTeamHP` | 🔵 | 订阅 `/team_hp`，写入黑板 `{team_hp}` |
| `ParseSentryBlackboard` | 🔴 | 解析 `team_outpost_hp` / `team_base_hp` 等派生变量 |

## 4. ParseSentryBlackboard 变量映射

> `ParseSentryBlackboard` 是**唯一的集中式解析器**，接收全部 12 个订阅话题的 raw msg，输出 **70+ 个**黑板变量。  
> 订阅节点仅存储原始消息；所有解析逻辑统一在 `ParseSentryBlackboard` 中完成。

---

### 4.1 原有 5 话题解析输出

| 来源话题 | 输出变量 |
|:---|:---|
| `/game_status` | `stage_remain`, `elapsed_time` |
| `/robot_status` | `hp_cur`, `hp_max`, `heat_cur`, `heat_max`, `ammo_allow`, `is_dead`, `is_disengaged`, `disengage_countdown`, `shooter_power_output` |
| `/rfid_status` | *(由条件节点直接解析)* |
| `/robot_position` | `pose_x`, `pose_y`, `pose_yaw`, `is_at_nav_goal`, `pose` |
| `/radar/enemy_tracks` | `has_target`, `best_target` |

### 4.2 🆕 7 个新话题解析输出

| 来源话题 | 输出变量 |
|:---|:---|
| `/sentry_decision_status` | `can_free_respawn`, `can_instant_respawn`, `instant_respawn_cost`, `current_posture`, `remote_ammo_count`, `remote_heal_count`, `exchanged_ammo_total`, `can_activate_energy` |
| `/robot_buff` | `buff_heal_rate`, `buff_cool_value`, `buff_defense_pct`, `buff_vulnerability_pct`, `buff_attack_pct` |
| `/projectile_allowance` | `fortress_ammo` |
| `/field_status` | `field_central_highland`, `field_ladder_highland`, `field_fortress`, `field_outpost_buff`, `field_base_buff`, `field_small_energy`, `field_big_energy` |
| `/enemy_mark` | `enemy_hero_vuln`, `enemy_engi_vuln`, `enemy_infantry3_vuln`, `enemy_infantry4_vuln`, `enemy_sentry_vuln` |
| `/team_positions` | *(广播给队友)* |
| `/team_hp` | `team_outpost_hp`, `team_base_hp` |

### 4.3 P1 额外输出变量

| 变量 | 类型 | 说明 |
|:---|:---|:---|
| `is_respawn_invincible` | `bool` | 复活无敌状态 |
| `respawn_invincible_remain_s` | `float` | 复活无敌剩余秒数 |
| `is_power_boosted` | `bool` | 功率提升状态 |
| `power_boost_remain_s` | `float` | 功率提升剩余秒数 |
| `cumulative_instant_count` | `int` | 累计立即复活次数 (inout) |

## 5. 主程序 RosNodeParams 配置

> 📄 文件：`rm_behavior_tree/rm_behavior_tree/src/rm_behavior_tree_rmuc.cpp`

改造前，所有 RMUC 订阅/发布插件共用一个 `params_rmuc`（绑定 `/rmuc` 话题）。  
改造后，**每个话题分配独立的 `RosNodeParams`**，各自拥有独立的 ROS 节点和默认话题名。

---

### 5.1 参数定义一览

```cpp
// ── 12 个输入订阅参数 ──

// 原有 5 个
BT::RosNodeParams params_game_status;
params_game_status.nh = std::make_shared<rclcpp::Node>("rmuc_game_status_io");
params_game_status.default_port_value = "/game_status";

BT::RosNodeParams params_robot_status;
params_robot_status.nh = std::make_shared<rclcpp::Node>("rmuc_robot_status_io");
params_robot_status.default_port_value = "/robot_status";

BT::RosNodeParams params_rfid_status;
params_rfid_status.nh = std::make_shared<rclcpp::Node>("rmuc_rfid_status_io");
params_rfid_status.default_port_value = "/rfid_status";

BT::RosNodeParams params_robot_position;
params_robot_position.nh = std::make_shared<rclcpp::Node>("rmuc_robot_position_io");
params_robot_position.default_port_value = "/robot_position";

BT::RosNodeParams params_radar;
params_radar.nh = std::make_shared<rclcpp::Node>("rmuc_radar_io");
params_radar.default_port_value = "/radar/enemy_tracks";

// 🆕 新增 7 个
BT::RosNodeParams params_sentry_decision;
params_sentry_decision.nh = std::make_shared<rclcpp::Node>("rmuc_sentry_decision_io");
params_sentry_decision.default_port_value = "/sentry_decision_status";

BT::RosNodeParams params_robot_buff;
params_robot_buff.nh = std::make_shared<rclcpp::Node>("rmuc_robot_buff_io");
params_robot_buff.default_port_value = "/robot_buff";

BT::RosNodeParams params_projectile_allowance;
params_projectile_allowance.nh = std::make_shared<rclcpp::Node>("rmuc_projectile_allowance_io");
params_projectile_allowance.default_port_value = "/projectile_allowance";

BT::RosNodeParams params_field_status;
params_field_status.nh = std::make_shared<rclcpp::Node>("rmuc_field_status_io");
params_field_status.default_port_value = "/field_status";

BT::RosNodeParams params_enemy_mark;
params_enemy_mark.nh = std::make_shared<rclcpp::Node>("rmuc_enemy_mark_io");
params_enemy_mark.default_port_value = "/enemy_mark";

BT::RosNodeParams params_team_positions;
params_team_positions.nh = std::make_shared<rclcpp::Node>("rmuc_team_positions_io");
params_team_positions.default_port_value = "/team_positions";

BT::RosNodeParams params_team_hp;
params_team_hp.nh = std::make_shared<rclcpp::Node>("rmuc_team_hp_io");
params_team_hp.default_port_value = "/team_hp";

// ── 3 个输出发布参数 ──
BT::RosNodeParams params_sentry_cmd;
params_sentry_cmd.nh = std::make_shared<rclcpp::Node>("rmuc_sentry_cmd_io");
params_sentry_cmd.default_port_value = "/sentry_cmd";

BT::RosNodeParams params_robot_ctrl;
params_robot_ctrl.nh = std::make_shared<rclcpp::Node>("rmuc_robot_ctrl_io");
params_robot_ctrl.default_port_value = "/robot_control";

BT::RosNodeParams params_nav_cmd;
params_nav_cmd.nh = std::make_shared<rclcpp::Node>("rmuc_nav_cmd_io");
params_nav_cmd.default_port_value = "/nav_control_cmd";
```

### 5.2 插件注册分组

```cpp
// ── Game Status 组 ──
factory.registerNodeType<RmucSubGameStatus>("SubGameStatus", params_game_status);

// ── Robot Status 组 ──
factory.registerNodeType<RmucSubRobotStatus>("SubRobotStatus", params_robot_status);
factory.registerNodeType<DetectRespawnAndSetRecovery>("DetectRespawnAndSetRecovery", params_robot_status);
factory.registerNodeType<WaitAndHeal>("WaitAndHeal", params_robot_status);

// ── RFID Status 组 ──
factory.registerNodeType<RmucSubRFIDStatus>("SubRFIDStatus", params_rfid_status);
factory.registerNodeType<MicroSearchSupplyCard>("MicroSearchSupplyCard", params_rfid_status);
factory.registerNodeType<IsSupplyCardDetected>("IsSupplyCardDetected", params_rfid_status);

// ── Robot Position 组 ──
factory.registerNodeType<RmucSubRobotPosition>("SubRobotPosition", params_robot_position);

// ── Radar 组 ──
factory.registerNodeType<SubRadarTracks>("SubRadarTracks", params_radar);

// ── 🆕 Sentry Decision Status 组 ──
factory.registerNodeType<RmucSubSentryDecisionStatus>("SubSentryDecisionStatus", params_sentry_decision);

// ── 🆕 Robot Buff 组 ──
factory.registerNodeType<RmucSubRobotBuff>("SubRobotBuff", params_robot_buff);

// ── 🆕 Projectile Allowance 组 ──
factory.registerNodeType<RmucSubProjectileAllowance>("SubProjectileAllowance", params_projectile_allowance);

// ── 🆕 Field Status 组 ──
factory.registerNodeType<RmucSubFieldStatus>("SubFieldStatus", params_field_status);

// ── 🆕 Enemy Mark 组 ──
factory.registerNodeType<RmucSubEnemyMark>("SubEnemyMark", params_enemy_mark);

// ── 🆕 Team Positions 组 ──
factory.registerNodeType<RmucSubTeamPositions>("SubTeamPositions", params_team_positions);

// ── 🆕 Team HP 组 ──
factory.registerNodeType<RmucSubTeamHP>("SubTeamHP", params_team_hp);

// ── 输出组 ──
factory.registerNodeType<SentryCmdMux>("SentryCmdMux", params_sentry_cmd);
factory.registerNodeType<RmucRobotControl>("RmucRobotControl", params_robot_ctrl);
factory.registerNodeType<RmucNavControlCmd>("RmucNavControlCmd", params_nav_cmd);
```

### 5.3 XML 端话题覆盖

虽然 `default_port_value` 已设置默认话题，XML 中仍可通过 `topic_name` 属性覆盖。  
以下列出全部 **15 个话题**（12 📥 订阅 + 3 📤 发布）的 XML 配置：

```xml
<!-- ═══ 📥 12 个输入订阅 (PerceptionAndBlackboard.xml) ═══ -->

<!-- 原有 5 个 -->
<SubGameStatus    topic_name="/game_status"              game_status="{game_status}" />
<SubRobotStatus   topic_name="/robot_status"             robot_status="{robot_status}" />
<SubRFIDStatus    topic_name="/rfid_status"              rfid_status="{rfid_status}" />
<SubRobotPosition topic_name="/robot_position"           ... is_at_nav_goal="{nav.at_goal}" />
<SubRadarTracks   topic_name="/radar/enemy_tracks"       radar_tracks="{radar_tracks}" />

<!-- 🆕 新增 7 个 -->
<SubSentryDecisionStatus topic_name="/sentry_decision_status"  sentry_decision_status="{sentry_decision_status}" />
<SubRobotBuff            topic_name="/robot_buff"               robot_buff="{robot_buff}" />
<SubProjectileAllowance  topic_name="/projectile_allowance"     projectile_allowance="{projectile_allowance}" />
<SubFieldStatus          topic_name="/field_status"             field_status="{field_status}" />
<SubEnemyMark            topic_name="/enemy_mark"               enemy_mark="{enemy_mark}" />
<SubTeamPositions        topic_name="/team_positions"           team_positions="{team_positions}" />
<SubTeamHP               topic_name="/team_hp"                  team_hp="{team_hp}" />

<!-- ═══ 📤 3 个输出发布 (分布在不同子树中) ═══ -->
<!-- CommandHub.xml -->
<SentryCmdMux     topic_name="/sentry_cmd"         ... />
<!-- 多个战术子树 (DeathAndRespawn / EngageCombat / PatrolAndScan 等) -->
<RmucRobotControl topic_name="/robot_control"      stop_gimbal_scan="..." chassis_spin="..." fire_enable="..." />
<!-- RespawnRecovery.xml -->
<RmucNavControlCmd topic_name="/nav_control_cmd"   cmd_type="..." emergency_stop="..." />
```

## 6. 架构说明

### 6.1 设计原则

| 原则 | 说明 |
|:---|:---|
| **绝对话题路径** | 所有话题名使用绝对路径（如 `/game_status`），避免命名空间混淆 |
| **订阅节点仅存储** | 12 个订阅节点只负责存储 raw msg 到黑板，不做任何解析 |
| **集中解析** | `ParseSentryBlackboard` 是唯一的集中式解析器，统一处理所有话题的派生变量 |
| **协议对齐** | 7 个新增话题对应 RMUC 2026 裁判系统协议新增数据帧 |

### 6.2 数据流示意

```
  裁判系统/传感器                    行为树黑板                       决策逻辑
  ─────────────              ─────────────────────              ───────────
  /game_status        ──→  RmucSubGameStatus        ──→ {game_status}    ─┐
  /robot_status       ──→  RmucSubRobotStatus       ──→ {robot_status}   ─┤
  /rfid_status        ──→  RmucSubRFIDStatus        ──→ {rfid_status}    ─┤
  /robot_position     ──→  RmucSubRobotPosition     ──→ {pose_x/y/yaw}  ─┤
  /radar/enemy_tracks ──→  SubRadarTracks           ──→ {radar_tracks}   ─┤
  /sentry_decision_status ──→ RmucSubSentryDecisionStatus ──→ {sentry_decision_status} ─┤
  /robot_buff         ──→  RmucSubRobotBuff         ──→ {robot_buff}     ─┤
  /projectile_allowance ──→ RmucSubProjectileAllowance ──→ {projectile_allowance} ─┤
  /field_status       ──→  RmucSubFieldStatus       ──→ {field_status}   ─┤
  /enemy_mark         ──→  RmucSubEnemyMark         ──→ {enemy_mark}     ─┤
  /team_positions     ──→  RmucSubTeamPositions     ──→ {team_positions} ─┤
  /team_hp            ──→  RmucSubTeamHP            ──→ {team_hp}        ─┤
                                                                          │
                                                    ParseSentryBlackboard ◄┘
                                                          │
                                                          ▼
                                                    70+ 派生黑板变量
                                                          │
                                                          ▼
                                                    条件节点 / 动作节点
```

## 7. 修改记录与踩坑备忘

### 📝 修改文件清单

| 类别 | 文件 / 路径 | 修改内容 |
|:---|:---|:---|
| **消息定义** | `rm_decision_interfaces/msg/RMUC/*.msg` (×15) | 新增 15 个拆分消息 (12 订阅 + 3 发布) |
| **接口构建** | `rm_decision_interfaces/CMakeLists.txt` | 添加 15 个 msg 到 `rosidl_generate_interfaces` |
| **插件头文件** | `rmuc_plugins/*.hpp` (×28+) | include 路径 + 类型名替换 (含 7 个新增订阅插件) |
| **插件源文件** | `rmuc_plugins/*.cpp` (×28+) | 类型名替换 |
| **主程序入口** | `rm_behavior_tree_rmuc.cpp` | 重写为 15 组独立 RosNodeParams |
| **XML 子树** | `PerceptionAndBlackboard.xml` | 12 个订阅者话题 + 新增输出端口 |
| **XML 子树** | `RespawnRecovery.xml` | topic_name 更新 + IsAtNavGoal 改为读 bool |
| **XML 主树** | `rmuc_2026.xml` | TreeNodesModel 默认值全部更新 |
| **测试脚本** | `test_rmuc_bt.py` | 改为 12 话题独立发布 |

---

### ⚠️ 踩坑记录

#### 1. 生成头文件路径是扁平的
```
❌ #include "rm_decision_interfaces/msg/rmuc/rmuc_game_status.hpp"
✅ #include "rm_decision_interfaces/msg/rmuc_game_status.hpp"
```
> `rosidl` 生成的头文件统一放在 `msg/` 目录下，不会保留源文件的子目录结构。

#### 2. `RMUCRFIDStatus` 的 snake_case 没有下划线
```
❌ rmuc_rfid_status.hpp
✅ rmucrfid_status.hpp
```
> ROS 2 的 CamelCase → snake_case 规则：`RMUC` → `rmuc`，`RFID` → `rfid`，中间**不插入**下划线。可用 `ros2 interface show` 验证。

#### 3. sed 批量替换导致双后缀
```
❌ RMUCGameStatusGameStatus  (已含后缀的文件被二次替换)
✅ RMUCGameStatus
```
> 对已完成部分替换的文件执行全局 `RMUC` → `RMUCGameStatus`，导致 `RMUCGameStatus` 被再次替换。解决：先检查是否已替换。

#### 4. `IsAtNavGoal` 类型冲突
```
原: InputPort<RMUCRobotPosition>("rfid_status")  ← 类型+key 均不匹配
改: InputPort<bool>("is_at_nav_goal")             ← 从 SubRobotPosition 输出的 bool
```
> `SubRobotPosition` 新增 `is_at_nav_goal` 输出端口，写入黑板 `{nav.at_goal}`；`IsAtNavGoal` 直接读取该 bool 值。

#### 5. XML `topic_name` 优先于 `default_port_value`
> 即使 C++ 中设置了 `params.default_port_value = "/robot_status"`，XML 中若残留 `topic_name="/rmuc"` 会覆盖默认值。**必须同步修改 XML**。

---

### ✅ 验证方式

```bash
# 1. 编译
colcon build --packages-select rm_decision_interfaces rm_behavior_tree

# 2. 启动行为树
ros2 run rm_behavior_tree rm_behavior_tree_rmuc \
  --ros-args -p style:=<path_to>/rmuc_2026.xml

# 3. 另一终端运行测试脚本
python3 test_rmuc_bt.py

# 4. 检查话题列表 (应显示 12 个订阅 + 3 个发布 = 15 个话题)
ros2 topic list | grep -E \
  "game_status|robot_status|rfid_status|robot_position|enemy_tracks|\
sentry_decision_status|robot_buff|projectile_allowance|field_status|\
enemy_mark|team_positions|team_hp|sentry_cmd|robot_control|nav_control_cmd"
```